# Experimentations
This notebook permits to run experimentations to MLFlow locally

In [1]:
import mlflow
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import pandas as pd
import os
from mlflow.tracking import MlflowClient
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

# check tracking URI
assert "MLFLOW_TRACKING_URI" in os.environ, "MLFLOW_TRACKING_URI non définie"

experiment_name = "2-xgboost"
seuil_utilisateur = 0.5 # pour optimiser F1-score 0.656, pour optimiser recall, mettre 0.3

client = MlflowClient()

exp = client.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(experiment_name)
else:
    exp_id = exp.experiment_id

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
import mlflow
import mlflow.xgboost
import matplotlib.pyplot as plt

# Load prepared data
df = pd.read_csv("../.data/modelisation/app_train.csv")
target_col = "TARGET"
X = df.drop(columns=[target_col])
y = df[target_col]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Prepare scoring
scoring = {
    "auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

# Define threshold
seuil_utilisateur = 0.5

# run MLflow
with mlflow.start_run(experiment_id=exp_id):
    mlflow.log_artifact("3-experimentation-xgboost.ipynb", artifact_path="notebooks")

    # Instantiate model with cost function (10x coeff)
    model = XGBClassifier(n_estimators=100, random_state=42, scale_pos_weight=10)

    # Cross-validation on training set
    cv_results = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=5,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    # Log params
    mlflow.log_params(model.get_params())

    # Fit final model on the whole training set (to evaluate on X_test)
    model.fit(X_train, y_train)
    
    # Predict probabilities on test set
    y_proba = model.predict_proba(X_test)[:, 1]

    # Apply threshold
    y_pred_utilisateur = (y_proba >= seuil_utilisateur).astype(int)
    precision_util = precision_score(y_test, y_pred_utilisateur)
    recall_util = recall_score(y_test, y_pred_utilisateur)
    f1_util = f1_score(y_test, y_pred_utilisateur)
    roc_auc = roc_auc_score(y_test, y_pred_utilisateur)
    
    # Log metrics with this threshold
    mlflow.log_metric("probability_threshold", round(seuil_utilisateur, 3))
    mlflow.log_metric("precision", round(precision_util, 3))
    mlflow.log_metric("recall", round(recall_util, 3))
    mlflow.log_metric("f1", round(f1_util, 3))
    mlflow.log_metric("auc", round(roc_auc, 3))

    # Log model
    try:
        mlflow.xgboost.log_model(model, name="model")
    except TypeError:
        # fallback if signature changed
        mlflow.xgboost.log_model(model, name="model")

/usr/local/lib/python3.13/site-packages/xgboost/sklearn.py:1028: UserWarning: [11:24:54] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2025/10/14 11:24:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run monumental-koi-28 at: http://mlflow:5000/#/experiments/367982218720826513/runs/ffddd2b4c0f9462faecd31454ab24c7e
🧪 View experiment at: http://mlflow:5000/#/experiments/367982218720826513
